# Grover on Caesar and Vigenère Ciphers

In this notebook, we will implement Grover's algorithm to solve a simple instance of the Caesar cipher problem. The Caesar cipher is a type of substitution cipher where each letter in the plaintext is shifted a certain number of places down the alphabet.

We will also implement Grover's algorithm for the Vigenère cipher, which is a more complex substitution cipher that uses a keyword to shift letters in the plaintext.

## Importing necessary libraries

In [1]:
import math

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import DraperQFTAdder, MCMTGate, ZGate
from qiskit_aer import AerSimulator

from src.algorithms.quantum_primitives.qpe import qpe

## Modifying the Grover class

In the previous [Grover Notebook](./grover_algorithm.ipynb), we discussed the standard implementation of Grover's algorithm and its limitations when applied to complex cryptographic scenarios. In this standard implementations of Grover's algorithm, the **Diffusion Operator** (which performs the inversion about the mean) acts globally over the entire quantum register. 

However, for advanced cryptanalysis like the **Vigenère cipher attack**, applying a global diffuser would entangle all the sub-keys together. For a 3-block key of 4 bits each, a global diffuser would operate over a combined search space of $N = 2^{12} = 4096$, forcing the algorithm to run ~50 iterations.

To circumvent this combinatorial explosion, we have modified the `Grover` class by introducing the `diffusion_registers` parameter. This architectural upgrade allows us to decouple the diffusion operator. Instead of one massive multi-controlled Z gate, the `_build_diffuser` method dynamically applies independent diffusers to specific sub-registers. 

This enables **Parallel Amplitude Amplification**, allowing us to search for the three independent key fragments concurrently in just 3 iterations.

In [2]:
class Grover:
    def __init__(
        self,
        oracle: QuantumCircuit,
        num_solutions: int = None,
        state_preparation: QuantumCircuit = None,
        search_space_size: int = None,
        diffusion_registers: list[list[int]] = None,
        simulator=AerSimulator(),
    ):
        """
        Initializes the Grover search engine.

        Args:
            oracle (QuantumCircuit): The quantum circuit implementing the oracle.
            num_solutions (int): The expected number of valid solutions.
                If None, it will be estimated using Quantum Counting.
            state_preparation (QuantumCircuit, optional): The heuristic circuit A.
                If None, standard H gates are applied.
            search_space_size (int, optional): The reduced search space N if a
                heuristic is used. Defaults to 2^n.
            diffusion_registers (list[list[int]], optional): The list of qubit registers
            to be used for the diffusion operator. Defaults to all qubits.
            simulator: The Qiskit backend/simulator to execute the circuit.
        """
        self.oracle = oracle
        self.num_qubits = oracle.num_qubits
        self.num_solutions = num_solutions
        self.simulator = simulator
        self.diffusion_registers = diffusion_registers

        # Default to global diffusion if no specific registers are provided
        if self.diffusion_registers is None:
            self.diffusion_registers = [list(range(self.num_qubits))]

        # Handle State Preparation (Standard Grover vs Amplitude Amplification)
        if state_preparation is None:
            # Standard Grover: H on all qubits
            qc = QuantumCircuit(self.num_qubits, name="Standard_H")
            qc.h(range(self.num_qubits))
            self.state_preparation = qc
            self.search_space_size = 2**self.num_qubits
        else:
            # Amplitude Amplification: Custom heuristic A
            self.state_preparation = state_preparation
            # Use provided reduced space size, or default to max
            self.search_space_size = search_space_size or (2**self.num_qubits)

        if self.num_solutions is not None:
            self.optimal_iterations = self._calculate_optimal_iterations()
        else:
            self.optimal_iterations = 0  # Will be updated after quantum counting

    def _calculate_optimal_iterations(self) -> int:
        """
        Dynamically calculates the optimal number of iterations to maximize
        the probability of measuring the target state.
        """
        N = self.search_space_size

        # Prevent math errors if solutions are >= to half the search space
        if self.num_solutions >= N / 2:
            return 0

        # Formula: floor((pi/4) * sqrt(N/M))
        theta = math.asin(math.sqrt(self.num_solutions / N))
        iterations = math.floor(math.pi / (4 * theta))

        return iterations

    def _build_diffuser(self) -> QuantumCircuit:
        """
        Constructs the generalized diffusion operator: A * (2|0><0| - I) * A_inv.
        Supports parallel amplitude amplification via multiple independent diffusers.
        If state_preparation is standard H, this naturally reduces to standard Grover.
        """
        qc = QuantumCircuit(self.num_qubits, name="Diffuser")

        qc.compose(self.state_preparation.inverse(), inplace=True)  # Apply A^(-1)

        for reg in self.diffusion_registers:
            qc.x(reg)
            # Apply multi-controlled Z gate only to qubits in the current diffusion reg
            num_controls = len(reg) - 1
            if num_controls > 0:
                qc.compose(MCMTGate(ZGate(), num_controls, 1), qubits=reg, inplace=True)
            else:
                qc.z(reg[0])  # Case of 1 single qubit

            qc.x(reg)
        qc.compose(self.state_preparation, inplace=True)  # Re-apply A

        # Add global phase of π per independent diffusion block for correct reflection
        qc.global_phase = math.pi * len(self.diffusion_registers)

        return qc

    def count_solutions(
        self, estimation_wires: int = 6, shots: int = 2048, debug: bool = False
    ) -> int:
        """
        Uses Quantum Phase Estimation to blindly estimate the number of solutions M.
        Updates the internal state (num_solutions and optimal_iterations) automatically.
        """
        if debug:
            print("\n--- QUANTUM COUNTING PHASE ---")
            print(f"Estimating solutions using {estimation_wires} precision qubits...")

        target_wires = self.num_qubits
        total_qubits = estimation_wires + target_wires
        counting_circ = QuantumCircuit(total_qubits, estimation_wires)

        # State Preparation: Apply heuristic A (or H) to the target register
        state_prep = QuantumCircuit(total_qubits)
        state_prep.compose(
            self.state_preparation,
            qubits=range(estimation_wires, total_qubits),
            inplace=True,
        )

        # Extract Grover Operator and inject into QPE
        grover_op = self.get_grover_operator()
        qpe_engine = qpe(estimation_wires, operator_u=grover_op)

        # Assemble and Measure
        counting_circ.compose(state_prep, inplace=True)
        counting_circ.compose(qpe_engine, inplace=True)
        counting_circ.measure(range(estimation_wires), range(estimation_wires))

        # Execute on Simulator
        compiled_circ = transpile(counting_circ, self.simulator)
        results = self.simulator.run(compiled_circ, shots=shots).result().get_counts()

        # Classical Post-Processing
        most_frequent_state = max(results, key=results.get)
        measured_integer = int(most_frequent_state, 2)

        N = self.search_space_size
        phase_fraction = measured_integer / (2**estimation_wires)
        theta = math.pi * phase_fraction

        M_raw = N * (math.sin(theta) ** 2)
        estimated_M = int(round(M_raw))

        if debug:
            print(f"Measured phase fraction: {phase_fraction:.4f}")
            print(f"Estimated solutions (M'): {estimated_M}")

        # Update internal class state
        self.num_solutions = estimated_M
        self.optimal_iterations = self._calculate_optimal_iterations()

        return estimated_M

    def get_grover_operator(self) -> QuantumCircuit:
        """
        Returns the combined Grover operator (Oracle + Diffuser) for a single iteration.
        This is particularly useful to export the operator for Quantum Counting.
        """
        grover_op = QuantumCircuit(self.num_qubits, name="Grover_Op")
        grover_op.compose(self.oracle, inplace=True)
        grover_op.compose(self._build_diffuser(), inplace=True)
        return grover_op

    def build_circuit(self, draw_circ: bool = False) -> QuantumCircuit:
        """
        Assembles the complete Grover's algorithm quantum circuit.
        """
        qr = QuantumRegister(self.num_qubits, name="q")
        cr = ClassicalRegister(self.num_qubits, name="c")
        qc = QuantumCircuit(qr, cr)

        qc.compose(self.state_preparation, inplace=True)  # Apply heuristic A or H

        # Apply the Grover operator the optimal number of times
        diffuser = self._build_diffuser()
        for _ in range(self.optimal_iterations):
            qc.compose(self.oracle, inplace=True)
            qc.compose(diffuser, inplace=True)
            qc.barrier()

        qc.measure(qr, cr)

        if draw_circ:
            qc.draw("mpl", style="iqp", fold=-1, idle_wires=False)

        return qc

    def search(self, shots: int = 1024, debug: bool = False) -> dict:
        """
        The main orchestrator method. Compiles and runs the circuit on the simulator.

        Returns:
            dict: The measurement counts.
        """
        if self.num_solutions is None:
            if debug:
                print("Number of solutions unknown. Running Quantum Counting...")
            self.count_solutions(debug=debug)

        if debug:
            print(f"Starting Grover's Algorithm for {self.num_qubits} qubits.")
            print(f"Expected solutions: {self.num_solutions}")
            print(f"Applying {self.optimal_iterations} optimal iterations...")

        qc = self.build_circuit(draw_circ=debug)

        if debug:
            print("Transpiling and executing...")

        compiled_circuit = transpile(qc, self.simulator)
        job = self.simulator.run(compiled_circuit, shots=shots)
        counts = job.result().get_counts()

        if debug:  # Sort and display the most frequent results
            sorted_counts = sorted(
                counts.items(), key=lambda item: item[1], reverse=True
            )
            print("\n--- Top Measurement Results ---")
            for state, count in sorted_counts[:5]:
                print(f"State: |{state}> -> {count} occurrences ({count / shots:.1%})")

        return counts

    @classmethod
    def execute(
        cls,
        oracle: QuantumCircuit,
        num_solutions: int = 1,
        shots: int = 1024,
        debug: bool = False,
    ) -> dict:
        """
        A convenience method to execute Grover's algorithm without explicitly creating
        an instance.

        Args:
            oracle (QuantumCircuit): The quantum circuit implementing the oracle.
            num_solutions (int): The expected number of valid solutions (marked states).
            shots (int): Number of shots for the simulation.
            debug (bool): If True, prints detailed debug information.

        Returns:
            dict: The measurement counts.
        """
        grover_instance = cls(oracle, num_solutions)
        return grover_instance.search(shots=shots, debug=debug)


## Caesar Cipher Quantum Cryptanalysis


The Caesar cipher encrypts by applying a single key $K$ to all plaintext elements: $C_i = (P_i + K) \pmod{16}$.
We will use two known plaintext-ciphertext pairs $(P_0, C_0)$ and $(P_1, C_1)$ to build the oracle.
The oracle uses a Quantum Fourier Transform Adder to perform the modular addition.

In [3]:
def caesar_state_prep(P0_val: int, P1_val: int, num_bits: int = 4) -> QuantumCircuit:
    K = QuantumRegister(num_bits, "k")
    P0 = QuantumRegister(num_bits, "p0")
    P1 = QuantumRegister(num_bits, "p1")
    C = QuantumRegister(2, "cout")
    qc = QuantumCircuit(K, P0, P1, C, name="State_Prep")

    # Superposition for the key K
    qc.h(K)

    # Encode Plaintext values in computational basis
    for i in range(num_bits):
        if (P0_val >> i) & 1:
            qc.x(P0[i])
        if (P1_val >> i) & 1:
            qc.x(P1[i])

    return qc


In [4]:
def caesar_oracle(C0_val: int, C1_val: int, num_bits: int = 4) -> QuantumCircuit:
    K = QuantumRegister(num_bits, "k")
    P0 = QuantumRegister(num_bits, "p0")
    P1 = QuantumRegister(num_bits, "p1")
    C = QuantumRegister(2, "cout")
    qc = QuantumCircuit(K, P0, P1, C, name="Oracle")

    # 1. Quantum Modular Adder (Add K to P0 and P1)
    adder = DraperQFTAdder(num_bits, kind="half")
    qc.append(adder, list(K) + list(P0) + [C[0]])
    qc.append(adder, list(K) + list(P1) + [C[1]])

    # 2. Check Condition (Are they equal to C0 and C1?)
    # Flip bits where ciphertext should be 0
    for i in range(num_bits):
        if not ((C0_val >> i) & 1):
            qc.x(P0[i])
        if not ((C1_val >> i) & 1):
            qc.x(P1[i])

    # 3. Phase Inversion (Multi-Controlled Z) on ALL plaintext qubits
    p_qubits = list(P0) + list(P1)
    qc.compose(MCMTGate(ZGate(), len(p_qubits) - 1, 1), qubits=p_qubits, inplace=True)

    # 4. Uncompute (Restore original state)
    for i in range(num_bits):
        if not ((C0_val >> i) & 1):
            qc.x(P0[i])
        if not ((C1_val >> i) & 1):
            qc.x(P1[i])

    qc.append(adder.inverse(), list(K) + list(P1) + [C[1]])
    qc.append(adder.inverse(), list(K) + list(P0) + [C[0]])

    return qc


In [5]:
# We define the known pairs:
# Let K = 5. If P0 = 2, C0 = (2+5)%16 = 7. If P1 = 14, C1 = (14+5)%16 = 3
P0_val, C0_val = 2, 7
P1_val, C1_val = 14, 3

num_bits = 4

print("--- CAESAR CIPHER ATTACK ---")
prep_circ = caesar_state_prep(P0_val, P1_val, num_bits)
oracle_circ = caesar_oracle(C0_val, C1_val, num_bits)

# Instantiate Grover targeting ONLY the Key register (qubits 0,1,2,3)
caesar_solver = Grover(
    oracle=oracle_circ,
    num_solutions=1,
    state_preparation=prep_circ,
    search_space_size=16,
    diffusion_registers=[[0, 1, 2, 3]],  # Diffuse over K only
)

caesar_results = caesar_solver.search(debug=True)


--- CAESAR CIPHER ATTACK ---
Starting Grover's Algorithm for 14 qubits.
Expected solutions: 1
Applying 3 optimal iterations...
Transpiling and executing...

--- Top Measurement Results ---
State: |00111000100101> -> 985 occurrences (96.2%)
State: |00111000101011> -> 8 occurrences (0.8%)
State: |00111000101010> -> 4 occurrences (0.4%)
State: |00111000100110> -> 4 occurrences (0.4%)
State: |00111000100100> -> 3 occurrences (0.3%)


In [6]:
def test_caesar_results(
    counts: dict, expected_k: int, expected_p0: int, expected_p1: int
):
    """
    Automated test to verify Caesar cipher quantum cryptanalysis results.
    Expected Qiskit bit string format (Left to Right): [Carry(2)] [P1(4)] [P0(4)] [K(4)]
    """
    print("\n--- CAESAR CIPHER VALIDATION TEST ---")
    most_frequent_state = max(counts, key=lambda state: counts[state])

    # Parse Little-Endian string
    carry_bits = most_frequent_state[0:2]
    p1_bits = most_frequent_state[2:6]
    p0_bits = most_frequent_state[6:10]
    k_bits = most_frequent_state[10:14]

    # Convert to decimal
    measured_k = int(k_bits, 2)
    measured_p0 = int(p0_bits, 2)
    measured_p1 = int(p1_bits, 2)

    print(f"Measured Key: {measured_k} | Expected: {expected_k}")
    print(f"Measured P0:  {measured_p0} | Expected: {expected_p0}")
    print(f"Measured P1:  {measured_p1} | Expected: {expected_p1}")
    print(f"Residual Carry: {carry_bits} (Should be '00')")

    assert measured_k == expected_k, "[-] Caesar Test Failed: Key mismatch."
    assert measured_p0 == expected_p0, (
        "[-] Caesar Test Failed: P0 uncomputation failed."
    )
    assert measured_p1 == expected_p1, (
        "[-] Caesar Test Failed: P1 uncomputation failed."
    )
    assert carry_bits == "00", "[-] Caesar Test Failed: Carry not clean."

    print(
        "[+] TEST PASSED: Caesar key successfully recovered and plaintexts uncomputed."
    )

In [7]:
test_caesar_results(caesar_results, expected_k=5, expected_p0=2, expected_p1=14)


--- CAESAR CIPHER VALIDATION TEST ---
Measured Key: 5 | Expected: 5
Measured P0:  2 | Expected: 2
Measured P1:  14 | Expected: 14
Residual Carry: 00 (Should be '00')
[+] TEST PASSED: Caesar key successfully recovered and plaintexts uncomputed.


### Caesar Cipher Analysis

The algorithm collapsed with $96.1\%$ probability into the state: `00111000100101`.
We parse this 14-bit string from right to left based on our register allocation:

*   **Key Register `K` (qubits 0-3):** `0101` $\rightarrow$ Decimal **5**. This is the exact secret key used to encrypt the messages!
*   **Plaintext `P0` (qubits 4-7):** `0010` $\rightarrow$ Decimal **2**. The uncompute step successfully restored the original $P_0$.
*   **Plaintext `P1` (qubits 8-11):** `1110` $\rightarrow$ Decimal **14**. The uncompute step successfully restored the original $P_1$.
*   **Carry Register `C` (qubits 12-13):** `00`. The quantum adder correctly uncomputed the carry bits back to their initial state.

**Result:** The key $K=5$ was fully recovered in just 3 quantum evaluations instead of classically iterating through the space.

## Vigenère Cipher Quantum Cryptanalysis

The Vigenère cipher uses a polyalphabetic key. We assume we do not know the key length.
We will recover three independent key elements $(K_0, K_1, K_2)$ for three plaintext elements.
To avoid entangling all key qubits and inflating the search space, we apply independent phase inversions for each block.

In [8]:
def vigenere_state_prep(P_vals):
    K = [QuantumRegister(4, f"k{i}") for i in range(3)]
    P = [QuantumRegister(4, f"p{i}") for i in range(3)]
    C = QuantumRegister(3, "cout")  # [NUEVO]: 3 qubits de acarreo
    qc = QuantumCircuit(*K, *P, C, name="State_Prep")

    for i in range(3):
        qc.h(K[i])
        for bit in range(4):
            if (P_vals[i] >> bit) & 1:
                qc.x(P[i][bit])

    return qc

In [9]:
def vigenere_oracle(C_vals):
    K = [QuantumRegister(4, f"k{i}") for i in range(3)]
    P = [QuantumRegister(4, f"p{i}") for i in range(3)]
    C = QuantumRegister(3, "cout")  # [NUEVO]: 3 qubits de acarreo
    qc = QuantumCircuit(*K, *P, C, name="Oracle")

    adder = DraperQFTAdder(4, kind="half")

    # 1. Independent Additions
    for i in range(3):
        qc.append(adder, list(K[i]) + list(P[i]) + [C[i]])

    # 2 & 3. Check Condition and Phase Inversions
    for i in range(3):
        for bit in range(4):
            if not ((C_vals[i] >> bit) & 1):
                qc.x(P[i][bit])

        qc.compose(MCMTGate(ZGate(), 3, 1), qubits=list(P[i]), inplace=True)

        for bit in range(4):
            if not ((C_vals[i] >> bit) & 1):
                qc.x(P[i][bit])

    # 4. Uncompute Additions
    for i in range(3):
        qc.append(adder.inverse(), list(K[i]) + list(P[i]) + [C[i]])

    return qc


In [10]:
P_vals = [4, 1, 9]
C_vals = [(4 + 2) % 16, (1 + 10) % 16, (9 + 15) % 16]  # [6, 11, 8]

print("\n--- VIGENÈRE CIPHER ATTACK (OPTIMIZED) ---")
prep_circ_vig = vigenere_state_prep(P_vals)
oracle_circ_vig = vigenere_oracle(C_vals)

# Instantiate Grover targeting the three sub-keys independently
# K0 is qubits 0-3, K1 is 4-7, K2 is 8-11
vigenere_solver = Grover(
    oracle=oracle_circ_vig,
    num_solutions=1,
    state_preparation=prep_circ_vig,
    search_space_size=16,  # Search space is mathematically 16 for each parallel search
    diffusion_registers=[
        [0, 1, 2, 3],  # Diffuser for K0
        [4, 5, 6, 7],  # Diffuser for K1
        [8, 9, 10, 11],  # Diffuser for K2
    ],
)

vigenere_results = vigenere_solver.search(debug=True)



--- VIGENÈRE CIPHER ATTACK (OPTIMIZED) ---
Starting Grover's Algorithm for 27 qubits.
Expected solutions: 1
Applying 3 optimal iterations...
Transpiling and executing...

--- Top Measurement Results ---
State: |000100100010100111110100010> -> 901 occurrences (88.0%)
State: |000100100010100000010100010> -> 8 occurrences (0.8%)
State: |000100100010100111110100000> -> 6 occurrences (0.6%)
State: |000100100010100111100100010> -> 6 occurrences (0.6%)
State: |000100100010100111110100101> -> 5 occurrences (0.5%)


### Vigenère Cipher Analysis

The parallel amplitude amplification collapsed with $89.3\%$ probability into the state: `000100100010100111110100010`.
Parsing this 27-bit string from right to left:

**1. Recovered Polyalphabetic Key:**
*   **Key `K0` (qubits 0-3):** `0010` $\rightarrow$ Decimal **2**.
*   **Key `K1` (qubits 4-7):** `1010` $\rightarrow$ Decimal **10**.
*   **Key `K2` (qubits 8-11):** `1111` $\rightarrow$ Decimal **15**.

**2. Restored Plaintexts:**
*   **Plaintext `P0` (qubits 12-15):** `0100` $\rightarrow$ Decimal **4**.
*   **Plaintext `P1` (qubits 16-19):** `0001` $\rightarrow$ Decimal **1**.
*   **Plaintext `P2` (qubits 20-23):** `1001` $\rightarrow$ Decimal **9**.

**3. Carries:**
*   **Carry `C` (qubits 24-26):** `000`. Clean uncomputation.

**Result:** The algorithm successfully circumvented the combinatorial explosion. Instead of entangling 12 key qubits and needing 50 iterations, our optimized parallel oracle found the independent key fragments $(K_0=2, K_1=10, K_2=15)$ concurrently in just 3 iterations.

In [11]:
def test_vigenere_results(counts: dict, expected_k: list, expected_p: list):
    """
    Automated test to verify Vigenère cipher quantum cryptanalysis results.
    Expected format: [Carry(3)] [P2(4)] [P1(4)] [P0(4)] [K2(4)] [K1(4)] [K0(4)]
    """
    print("\n--- VIGENÈRE CIPHER VALIDATION TEST ---")
    most_frequent_state = max(counts, key=lambda state: counts[state])

    # Parse Little-Endian string (27 bits)
    carry_bits = most_frequent_state[0:3]
    p_bits = [
        most_frequent_state[11:15],
        most_frequent_state[7:11],
        most_frequent_state[3:7],
    ]
    k_bits = [
        most_frequent_state[23:27],
        most_frequent_state[19:23],
        most_frequent_state[15:19],
    ]

    measured_k = [int(kb, 2) for kb in k_bits]
    measured_p = [int(pb, 2) for pb in p_bits]

    print(f"Measured Keys: {measured_k} | Expected: {expected_k}")
    print(f"Measured Plaintexts: {measured_p} | Expected: {expected_p}")
    print(f"Residual Carry: {carry_bits} (Should be '000')")

    assert measured_k == expected_k, "[-] Vigenère Test Failed: Keys mismatch."
    assert measured_p == expected_p, "[-] Vigenère Test Failed: Plaintexts mismatch."
    assert carry_bits == "000", "[-] Vigenère Test Failed: Carry not clean."

    print("[+] TEST PASSED: Vigenère sub-keys successfully recovered in parallel.")


In [12]:
test_vigenere_results(vigenere_results, expected_k=[2, 10, 15], expected_p=[4, 1, 9])


--- VIGENÈRE CIPHER VALIDATION TEST ---
Measured Keys: [2, 10, 15] | Expected: [2, 10, 15]
Measured Plaintexts: [4, 1, 9] | Expected: [4, 1, 9]
Residual Carry: 000 (Should be '000')
[+] TEST PASSED: Vigenère sub-keys successfully recovered in parallel.
